# SmolLM2-135M: A Forward Pass by Hand, from Tokens to a KV-Cache Decode Step

This notebook opens up SmolLM2-135M, a small model with the Llama architecture, and rebuilds its forward pass by hand, checking against Hugging Face along the way. It ends by running the last layer the way generation with a **KV cache** does: cached K/V for the earlier tokens, and fresh Q/K/V computed only for the new token.

It has six parts:
1. **Load the model and read its config.** Check the key dimensions: hidden size, heads, KV heads (GQA), head dim.
2. **Read the source code.** Print the Hugging Face implementation of attention, GQA `repeat_kv`, the decoder layer and the MLP.
3. **Rebuild layer 0 attention by hand.** Embeddings → RMSNorm → Q/K/V → heads → RoPE → GQA `repeat_kv` → scores → causal softmax → weighted V → `o_proj`, checked against HF.
4. **Finish block 0.** Residual → RMSNorm → SwiGLU MLP → residual gives block 0's output.
5. **Layers 1–28.** Run them with the model's own `layer.forward()`.
6. **Layer 29 as a KV-cache decode step.** Reuse cached K/V for every token except the last, compute Q/K/V only for the last token, then go through the final norm and LM head to predict the next token. The logits match the baseline.

---
## 1. Setup: load the model

Load the tokenizer and model in `float32` and switch to eval mode. Printing the module tree shows 30 `LlamaDecoderLayer`s. Note that `k_proj` / `v_proj` output only **192** features while `q_proj` outputs **576**. That gap is Grouped-Query Attention.

**Prompt length:** the prompt is set once, in the baseline cell of Section 3. Step 1 defines `T` (prompt length), `last = T - 1` and `last_tok`, and every later cell uses these, so any prompt length works. Shapes in the notes are written with `T`. The concrete numbers quoted in the notes come from two example prompts:

| Prompt | T | Last token | Predicted next token | Max logit diff (hand-built vs. HF) |
|---|---|---|---|---|
| `"one two three four five six"` (counting) | 6 | `six` | `' seven'` (id 4962) | ≈ 3.2e-5 |
| `"The quick brown fox jumps over the lazy"` (fox) | 8 | `lazy` | `' dog'` (id 2767) | ≈ 3.4e-5 |

Your own outputs will differ if you use another prompt, but the shapes and the checks hold for any length.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
)

model.eval()

print(model)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

### Key architecture numbers

Read the config values used in the rest of the notebook:
- `hidden_size = 576`, split into `9` attention heads × `head_dim = 64`
- `num_key_value_heads = 3`, so every KV head is shared by 9 / 3 = **3 query heads** (GQA)
- That is why K/V projections are `3 × 64 = 192` wide. It also makes the KV cache 3× smaller than with full multi-head attention.

In [2]:
config = model.config

print("Model type:          ", config.model_type)
print("Hidden size:         ", config.hidden_size)
print("Layers:              ", config.num_hidden_layers)
print("Attention heads:     ", config.num_attention_heads)
print("KV heads:            ", config.num_key_value_heads)
print("Head dimension:      ", config.head_dim)
print("Intermediate size:   ", config.intermediate_size)
print("Vocab size:          ", config.vocab_size)

Model type:           llama
Hidden size:          576
Layers:               30
Attention heads:      9
KV heads:             3
Head dimension:       64
Intermediate size:    1536
Vocab size:           49152


---
## 2. Read the Hugging Face implementation

Before rebuilding anything by hand, print the actual source code the model runs. Doing this first makes it easy to match each manual step in Section 3 to a line of library code.

### Attention forward (`LlamaAttention.forward`)
This is the core path: project to Q/K/V, reshape into heads, apply RoPE, **update the KV cache** (`past_key_values.update`), then call the attention kernel.

In [3]:

import inspect

attn = model.model.layers[0].self_attn

print(inspect.getsource(attn.forward))

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        attention_mask: torch.Tensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.head_dim)

        query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)

        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_values is not None:
            key_states, value_states = past_key_values.update(key_states, value_states, self.layer_idx)

        attention_interface: C

### The attention kernel (`eager_attention_forward`)
The plain PyTorch version of scaled dot-product attention: expand K/V for GQA, compute `QKᵀ · scale`, add the causal mask, apply softmax, multiply by V.

In [4]:
import inspect
from transformers.models.llama import modeling_llama

print(inspect.getsource(modeling_llama.eager_attention_forward))

def eager_attention_forward(
    module: nn.Module,
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    attention_mask: torch.Tensor | None,
    scaling: float,
    dropout: float = 0.0,
    **kwargs: Unpack[TransformersKwargs],
):
    key_states = repeat_kv(key, module.num_key_value_groups)
    value_states = repeat_kv(value, module.num_key_value_groups)

    attn_weights = torch.matmul(query, key_states.transpose(2, 3)) * scaling
    if attention_mask is not None:
        attn_weights = attn_weights + attention_mask

    attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query.dtype)
    attn_weights = nn.functional.dropout(attn_weights, p=dropout, training=module.training)
    attn_output = torch.matmul(attn_weights, value_states)
    attn_output = attn_output.transpose(1, 2).contiguous()

    return attn_output, attn_weights



### GQA helper (`repeat_kv`)
Repeats each of the 3 KV heads `n_rep = 3` times so they line up with the 9 query heads. Only the un-repeated K/V (3 heads) is stored in the cache. The expansion happens again at compute time.

In [5]:
import inspect
from transformers.models.llama import modeling_llama

print(inspect.getsource(modeling_llama.repeat_kv))

def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    This is the equivalent of torch.repeat_interleave(x, dim=1, repeats=n_rep). The hidden states go from (batch,
    num_key_value_heads, seqlen, head_dim) to (batch, num_attention_heads, seqlen, head_dim)
    """
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)



### Decoder layer (`LlamaDecoderLayer.forward`)
The pre-norm residual structure: `x + Attn(RMSNorm(x))`, then `x + MLP(RMSNorm(x))`.

In [6]:
import inspect

layer = model.model.layers[0]
print(inspect.getsource(layer.forward))

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        position_ids: torch.LongTensor | None = None,
        past_key_values: Cache | None = None,
        use_cache: bool | None = False,
        position_embeddings: tuple[torch.Tensor, torch.Tensor] | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> torch.Tensor:
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        # Self Attention
        hidden_states, _ = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            position_embeddings=position_embeddings,
            **kwargs,
        )
        hidden_states = residual + hidden_states

        # Fully Connected
        residual = hidden_states
        hidden_states = self.post_att

### Feed-forward (`LlamaMLP.forward`)
A SwiGLU MLP: `down_proj(act(gate_proj(x)) * up_proj(x))`, expanding 576 → 1536 → 576.

In [7]:
import inspect

print(inspect.getsource(layer.mlp.forward))

    def forward(self, x):
        down_proj = self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
        return down_proj



---
## 3. Rebuild layer 0 step by step

### Baseline: a full forward pass
Set the prompt here (this is the only place to change it) and run the whole model once as the reference. The logits have shape `[batch, T, vocab] = [1, T, 49152]`, and the next-token prediction comes from the last position only. The counting prompt predicts `' seven'` and the fox prompt predicts `' dog'`. To switch, comment one `text = ...` line in or out.

In [8]:
# Change the prompt here. The rest of the notebook adapts to its length.
#text = "one two three four five six"
text = "The quick brown fox jumps over the lazy"

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

next_token_id = outputs.logits[:, -1, :].argmax(dim=-1)

print("Input:", text)
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Predicted next token:", tokenizer.decode(next_token_id))
print("Logits shape:", outputs.logits.shape)

Input: The quick brown fox jumps over the lazy
Tokens: ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy']
Predicted next token:  dog
Logits shape: torch.Size([1, 8, 49152])


### Step 1: Token embeddings
Tokenize the same prompt and look up the embedding for each token. The result is the input `hidden_states` to layer 0, with shape `[1, T, 576]`.

This cell also defines the length variables used in the rest of the notebook: `T` (number of tokens), `last = T - 1` (index of the last token) and `last_tok` (its text, used in print labels).

In [9]:
# Reuses `text` from the baseline cell above
inputs = tokenizer(text, return_tensors="pt")

T = inputs["input_ids"].shape[1]   # prompt length in tokens
last = T - 1                       # index of the last token (the "new" token in Section 6)
last_tok = tokenizer.decode(inputs["input_ids"][0, last])

with torch.no_grad():
    hidden_states = model.model.embed_tokens(inputs["input_ids"])

print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Sequence length T:", T, "| last token:", repr(last_tok))
print("Embedding shape:", hidden_states.shape)
print("First token embedding:", hidden_states[0, 0, :10])

Tokens: ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy']
Sequence length T: 8 | last token: ' lazy'
Embedding shape: torch.Size([1, 8, 576])
First token embedding: tensor([ 0.1177,  0.0199, -0.0942,  0.0781,  0.0869, -0.0376,  0.0508,  0.0693,
        -0.0272,  0.0029])


### Step 2: Input RMSNorm
Apply layer 0's `input_layernorm` (RMSNorm). The shape stays the same, but each token vector is rescaled to unit RMS and multiplied by the learned weight.

In [10]:
block0 = model.model.layers[0]
with torch.no_grad():
    normed_states = block0.input_layernorm(hidden_states)

print("Before RMSNorm:", hidden_states.shape)
print("After RMSNorm :", normed_states.shape)

print("\nFirst token before:")
print(hidden_states[0, 0, :10])

print("\nFirst token after:")
print(normed_states[0, 0, :10])

Before RMSNorm: torch.Size([1, 8, 576])
After RMSNorm : torch.Size([1, 8, 576])

First token before:
tensor([ 0.1177,  0.0199, -0.0942,  0.0781,  0.0869, -0.0376,  0.0508,  0.0693,
        -0.0272,  0.0029])

First token after:
tensor([ 0.0270,  0.0045,  0.0421, -0.0319, -0.0294,  0.0128,  0.0153, -0.0210,
         0.0532,  0.0006])


### Step 3: Q, K, V projections
Project the normalized states with `q_proj`, `k_proj` and `v_proj`. Q is 576-wide (9 heads) but K and V are only 192-wide (3 heads). **These K and V vectors are what gets cached** for every token in every layer.

In [11]:
attn = block0.self_attn

with torch.no_grad():
    q = attn.q_proj(normed_states)
    k = attn.k_proj(normed_states)
    v = attn.v_proj(normed_states)

print("Q:", q.shape)
print("K:", k.shape)
print("V:", v.shape)

print("\nFirst token Q, first 10 values:")
print(q[0, 0, :10])

print("\nFirst token K, first 10 values:")
print(k[0, 0, :10])

print("\nFirst token V, first 10 values:")
print(v[0, 0, :10])

Q: torch.Size([1, 8, 576])
K: torch.Size([1, 8, 192])
V: torch.Size([1, 8, 192])

First token Q, first 10 values:
tensor([-1.2616, -0.7086, -0.8346, -1.0988,  0.7617, -0.9179,  0.4594,  0.7733,
        -0.2876, -0.9992])

First token K, first 10 values:
tensor([-1.2127,  0.1643, -0.2073, -0.4753,  0.3401, -0.0587, -0.3797,  0.8676,
        -0.0135, -0.2653])

First token V, first 10 values:
tensor([-0.0249,  0.0173, -0.0370,  0.0004,  0.0181, -0.0004,  0.0636, -0.0157,
         0.0171, -0.0144])


### Step 4: Split into heads
Repeat the reshaping done in `LlamaAttention.forward`: `view(..., -1, head_dim)` then `transpose(1, 2)`, giving `[batch, heads, seq_len, head_dim]`.
- Q → `[1, 9, T, 64]`
- K, V → `[1, 3, T, 64]`

In [12]:
attn = block0.self_attn

with torch.no_grad():
    input_shape = normed_states.shape[:-1]
    hidden_shape = (*input_shape, -1, attn.head_dim)

    query_states = attn.q_proj(normed_states).view(hidden_shape).transpose(1, 2)
    key_states = attn.k_proj(normed_states).view(hidden_shape).transpose(1, 2)
    value_states = attn.v_proj(normed_states).view(hidden_shape).transpose(1, 2)

print("Q:", query_states.shape)
print("K:", key_states.shape)
print("V:", value_states.shape)

Q: torch.Size([1, 9, 8, 64])
K: torch.Size([1, 3, 8, 64])
V: torch.Size([1, 3, 8, 64])


### Step 5: Compute RoPE cos/sin
Rotary position embeddings encode each token's position by rotating Q and K. Here we get the `cos` / `sin` tables for positions `0 … T-1` from `model.model.rotary_emb`. Each table is `[1, T, 64]`: one row per position, one value per head dimension.

Compare the two positions printed:
- **Position 0**: `cos = 1`, `sin = 0`. This is the identity rotation, so the first token is left unchanged.
- **Position 1**: real rotation angles. The first dimension turns by 1 radian (`cos ≈ 0.54`, `sin ≈ 0.84`), and later dimensions turn by smaller and smaller angles.

In [13]:
# Generate the actual RoPE cos/sin values for every token position

position_ids = torch.arange(
     inputs["input_ids"].shape[1],
    device=query_states.device
).unsqueeze(0)

with torch.no_grad():
    cos, sin = model.model.rotary_emb(
        query_states,
        position_ids
    )

print("position_ids:")
print(position_ids)

print("\ncos shape:", cos.shape)
print("sin shape:", sin.shape)

print("\ncos - first token - first 10 values:")
print(cos[0, 0, :10])

print("\nsin - first token - first 10 values:")
print(sin[0, 0, :10])

print("\ncos - second token - first 10 values:")
print(cos[0, 1, :10])

print("\nsin - second token - first 10 values:")
print(sin[0, 1, :10])

position_ids:
tensor([[0, 1, 2, 3, 4, 5, 6, 7]])

cos shape: torch.Size([1, 8, 64])
sin shape: torch.Size([1, 8, 64])

cos - first token - first 10 values:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

sin - first token - first 10 values:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

cos - second token - first 10 values:
tensor([0.5403, 0.7662, 0.8838, 0.9428, 0.9720, 0.9863, 0.9933, 0.9968, 0.9984,
        0.9992])

sin - second token - first 10 values:
tensor([0.8415, 0.6426, 0.4679, 0.3333, 0.2349, 0.1647, 0.1152, 0.0805, 0.0562,
        0.0392])


### Step 6: Apply RoPE to Q and K
Rotate Q and K with `apply_rotary_pos_emb`. The shapes do not change. V is **not** rotated. The KV cache stores K *after* RoPE, so cached keys already carry their position.

The output shows both cases:
- **Token 0**: the values before and after are identical, because position 0 is the identity rotation.
- **Token 1**: the values change. HF's `rotate_half` pairs dimension *i* with dimension *i + 32*, so each value printed here is mixed with a partner from the second half of the head vector.

In [14]:
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb

with torch.no_grad():
    rotated_q, rotated_k = apply_rotary_pos_emb(
        query_states,
        key_states,
        cos,
        sin,
    )

print("Original Q shape:", query_states.shape)
print("Rotated Q shape :", rotated_q.shape)

print("Original K shape:", key_states.shape)
print("Rotated K shape :", rotated_k.shape)

print("\nQ, first token, first 10 values BEFORE RoPE:")
print(query_states[0, 0, 0, :10])

print("\nQ, first token, first 10 values AFTER RoPE:")
print(rotated_q[0, 0, 0, :10])

print("\nK, first token, first 10 values BEFORE RoPE:")
print(key_states[0, 0, 0, :10])

print("\nK, first token, first 10 values AFTER RoPE:")
print(rotated_k[0, 0, 0, :10])

print("\nQ, second token, first 10 values BEFORE RoPE:")
print(query_states[0, 0, 1, :10])

print("\nQ, second token, first 10 values AFTER RoPE:")
print(rotated_q[0, 0, 1, :10])

print("\nK, second token, first 10 values BEFORE RoPE:")
print(key_states[0, 0, 1, :10])

print("\nK, second token, first 10 values AFTER RoPE:")
print(rotated_k[0, 0, 1, :10])

Original Q shape: torch.Size([1, 9, 8, 64])
Rotated Q shape : torch.Size([1, 9, 8, 64])
Original K shape: torch.Size([1, 3, 8, 64])
Rotated K shape : torch.Size([1, 3, 8, 64])

Q, first token, first 10 values BEFORE RoPE:
tensor([-1.2616, -0.7086, -0.8346, -1.0988,  0.7617, -0.9179,  0.4594,  0.7733,
        -0.2876, -0.9992])

Q, first token, first 10 values AFTER RoPE:
tensor([-1.2616, -0.7086, -0.8346, -1.0988,  0.7617, -0.9179,  0.4594,  0.7733,
        -0.2876, -0.9992])

K, first token, first 10 values BEFORE RoPE:
tensor([-1.2127,  0.1643, -0.2073, -0.4753,  0.3401, -0.0587, -0.3797,  0.8676,
        -0.0135, -0.2653])

K, first token, first 10 values AFTER RoPE:
tensor([-1.2127,  0.1643, -0.2073, -0.4753,  0.3401, -0.0587, -0.3797,  0.8676,
        -0.0135, -0.2653])

Q, second token, first 10 values BEFORE RoPE:
tensor([-0.2393,  0.0119,  0.0863, -0.9062,  1.0091, -0.9252,  0.3133,  0.4359,
         0.6476, -0.1731])

Q, second token, first 10 values AFTER RoPE:
tensor([ 0.029

### Check: why token 0 is unchanged
Put the first query head's values for token 0 next to its `cos`/`sin`. With `cos = 1, sin = 0`, the rotation `q·cos + rotate_half(q)·sin` gives back exactly `q`.

In [15]:
# Inspect one Q head and the RoPE parameters used for token 0

q0 = query_states[0, 0, 0]
q0_rot = rotated_q[0, 0, 0]

print("Original Q first 8:")
print(q0[:8])

print("\nRotated Q first 8:")
print(q0_rot[:8])

print("\ncos first 8:")
print(cos[0, 0, :8])

print("\nsin first 8:")
print(sin[0, 0, :8])

Original Q first 8:
tensor([-1.2616, -0.7086, -0.8346, -1.0988,  0.7617, -0.9179,  0.4594,  0.7733])

Rotated Q first 8:
tensor([-1.2616, -0.7086, -0.8346, -1.0988,  0.7617, -0.9179,  0.4594,  0.7733])

cos first 8:
tensor([1., 1., 1., 1., 1., 1., 1., 1.])

sin first 8:
tensor([0., 0., 0., 0., 0., 0., 0., 0.])


### A closer look at the RoPE frequencies (position 1)
At position *p*, dimension pair *i* is rotated by the angle `p · θᵢ`, where `θᵢ = base^(−2i/64)`. At position 1 the angle is just θᵢ, so this row shows the frequencies directly. The first pair rotates fast (θ₀ = 1 rad, `cos ≈ 0.54`) and later pairs rotate more and more slowly (`cos → 1`, `sin → 0`). Fast-rotating dimensions encode fine-grained, local position. Slow ones change little over short distances and carry long-range position.

In [16]:
print("Position 1 cos, first 10:")
print(cos[0, 1, :10])

print("\nPosition 1 sin, first 10:")
print(sin[0, 1, :10])

Position 1 cos, first 10:
tensor([0.5403, 0.7662, 0.8838, 0.9428, 0.9720, 0.9863, 0.9933, 0.9968, 0.9984,
        0.9992])

Position 1 sin, first 10:
tensor([0.8415, 0.6426, 0.4679, 0.3333, 0.2349, 0.1647, 0.1152, 0.0805, 0.0562,
        0.0392])


### Step 7: GQA shapes before `repeat_kv`
Set up for the attention kernel. After RoPE, Q still has 9 heads while K and V have 3. `num_key_value_groups = 9 / 3 = 3` is the `n_rep` factor `repeat_kv` will use to expand K/V to 9 heads. **The cache holds the 3-head K/V shown here**, not the expanded version, and that is where GQA saves memory.

In [17]:
attn = block0.self_attn

print("Before repeat_kv:")
print("Q:", rotated_q.shape)
print("K:", rotated_k.shape)
print("V:", value_states.shape)

print("\nnum_attention_heads:", attn.config.num_attention_heads)
print("num_key_value_heads:", attn.config.num_key_value_heads)
print("num_key_value_groups:", attn.num_key_value_groups)

Before repeat_kv:
Q: torch.Size([1, 9, 8, 64])
K: torch.Size([1, 3, 8, 64])
V: torch.Size([1, 3, 8, 64])

num_attention_heads: 9
num_key_value_heads: 3
num_key_value_groups: 3


### Step 8: Expand K/V with `repeat_kv`
Call the same `repeat_kv` helper the model uses, with `n_rep = num_key_value_groups = 3`. K and V go from 3 heads to 9, `[1, 3, T, 64] → [1, 9, T, 64]`, so each query head now has a matching K/V head. This expansion happens at compute time only. The cache keeps the smaller 3-head version.

In [18]:
from transformers.models.llama.modeling_llama import repeat_kv

with torch.no_grad():
    repeated_k = repeat_kv(
        rotated_k,
        attn.num_key_value_groups
    )

    repeated_v = repeat_kv(
        value_states,
        attn.num_key_value_groups
    )

print("K before:", rotated_k.shape)
print("K after :", repeated_k.shape)

print("\nV before:", value_states.shape)
print("V after :", repeated_v.shape)

K before: torch.Size([1, 3, 8, 64])
K after : torch.Size([1, 9, 8, 64])

V before: torch.Size([1, 3, 8, 64])
V after : torch.Size([1, 9, 8, 64])


### Check: which query heads share which KV head
Confirm that the expanded heads are exact copies, arranged in contiguous blocks:
- KV head 0 → query heads 0, 1, 2
- KV head 1 → query heads 3, 4, 5
- KV head 2 → query heads 6, 7, 8

This is the `repeat_interleave` layout described in the `repeat_kv` docstring. Each group of 3 query heads attends over the same keys and values.

In [19]:
# Compare the repeated K heads

print("K head 0 vs repeated heads 0,1,2:")
print(torch.allclose(repeated_k[0, 0], rotated_k[0, 0]))
print(torch.allclose(repeated_k[0, 1], rotated_k[0, 0]))
print(torch.allclose(repeated_k[0, 2], rotated_k[0, 0]))

print("\nK head 1 vs repeated heads 3,4,5:")
print(torch.allclose(repeated_k[0, 3], rotated_k[0, 1]))
print(torch.allclose(repeated_k[0, 4], rotated_k[0, 1]))
print(torch.allclose(repeated_k[0, 5], rotated_k[0, 1]))

print("\nK head 2 vs repeated heads 6,7,8:")
print(torch.allclose(repeated_k[0, 6], rotated_k[0, 2]))
print(torch.allclose(repeated_k[0, 7], rotated_k[0, 2]))
print(torch.allclose(repeated_k[0, 8], rotated_k[0, 2]))

K head 0 vs repeated heads 0,1,2:
True
True
True

K head 1 vs repeated heads 3,4,5:
True
True
True

K head 2 vs repeated heads 6,7,8:
True
True
True


### Step 9: Attention scores `Q · Kᵀ · scale`
Take the dot product of every query with every key, per head, scaled by `attn.scaling = 1/√head_dim = 1/√64 = 0.125`. The result is `[batch, heads, q_len, k_len] = [1, 9, T, T]`: one T×T score matrix per head. Row *i* holds how strongly token *i*'s query matches each token's key.

**The causal mask has not been applied yet.** The upper triangle, where earlier tokens score later tokens, still contains real values. The next steps are to add the mask and apply softmax.

In [20]:
# Attention scores: Q @ K^T

scaling = attn.scaling

with torch.no_grad():
    attn_scores = torch.matmul(
        rotated_q,
        repeated_k.transpose(2, 3)
    ) * scaling

print("Q shape:", rotated_q.shape)
print("K shape:", repeated_k.shape)
print("K^T shape:", repeated_k.transpose(2, 3).shape)

print("\nAttention scores shape:", attn_scores.shape)

print("\nHead 0 attention scores:")
print(attn_scores[0, 0])

Q shape: torch.Size([1, 9, 8, 64])
K shape: torch.Size([1, 9, 8, 64])
K^T shape: torch.Size([1, 9, 64, 8])

Attention scores shape: torch.Size([1, 9, 8, 8])

Head 0 attention scores:
tensor([[4.9444, 3.8685, 2.5183, 1.9366, 2.3226, 3.4930, 4.1557, 2.9867],
        [1.6060, 2.2611, 1.8199, 1.5398, 1.1040, 0.4581, 2.0446, 1.5213],
        [1.3781, 2.1517, 1.6999, 1.3851, 0.9033, 0.5583, 1.0480, 1.3239],
        [0.8915, 2.0945, 1.7905, 1.7303, 1.0701, 0.6290, 0.9024, 1.0881],
        [0.4483, 1.0382, 1.0026, 1.1776, 1.0709, 0.1449, 0.3307, 0.5380],
        [2.2735, 2.1446, 2.2373, 2.2774, 2.5950, 0.3950, 2.7864, 3.2711],
        [2.9127, 2.3247, 1.4561, 1.2359, 1.5532, 2.1701, 4.3129, 2.4745],
        [0.9512, 1.5161, 0.9081, 0.6172, 1.0501, 1.3023, 0.5608, 1.7355]])


### Step 10: Causal mask + softmax
Add the causal mask to the raw scores. `torch.triu(..., diagonal=1)` marks every *future* position, and those scores are set to `-inf`. Softmax turns each row into a probability distribution over the current and earlier tokens, and `-inf` becomes exactly 0.

Things to notice:
- **Row 0** (the first token) can only see itself, so it always gets probability **1.0**, whatever the prompt.
- **Each head learns its own pattern, and the patterns depend on the prompt.** For the last query:
  - *Counting prompt (`six`):* head 0 spreads attention evenly, with every weight around 0.1–0.2. Head 1 is sharper: it gives little weight to the first token and focuses on recent tokens.
  - *Fox prompt (`lazy`):* head 0 is again fairly spread out (0.07–0.22). Head 1 is more focused, putting about 0.34 on `lazy` itself and about 0.20 on `quick`. Also, in both heads the query `the` puts about 0.5–0.6 of its attention on itself.

This lower-triangular structure is what makes the KV cache possible. A token's K/V never depend on later tokens, so they can be computed once and reused.

In [21]:
# Apply causal mask, then softmax

with torch.no_grad():
    seq_len = attn_scores.shape[-1]

    # Upper-triangular positions are future tokens
    causal_mask = torch.triu(
        torch.ones(seq_len, seq_len, device=attn_scores.device, dtype=torch.bool),
        diagonal=1
    )

    masked_scores = attn_scores.masked_fill(causal_mask, float("-inf"))

    attention_probs = torch.softmax(masked_scores, dim=-1)

print("Masked scores, head 0:")
print(masked_scores[0, 0])

print("\nAttention probabilities, head 0:")
print(attention_probs[0, 0])

print(f"\nAttention probabilities for last query {last_tok!r}:")
print(attention_probs[0, 0, last])

print("Masked scores, head 1:")
print(masked_scores[0, 1])

print("\nAttention probabilities, head 1:")
print(attention_probs[0, 1])

print(f"\nAttention probabilities for last query {last_tok!r}:")
print(attention_probs[0, 1, last])

Masked scores, head 0:
tensor([[4.9444,   -inf,   -inf,   -inf,   -inf,   -inf,   -inf,   -inf],
        [1.6060, 2.2611,   -inf,   -inf,   -inf,   -inf,   -inf,   -inf],
        [1.3781, 2.1517, 1.6999,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.8915, 2.0945, 1.7905, 1.7303,   -inf,   -inf,   -inf,   -inf],
        [0.4483, 1.0382, 1.0026, 1.1776, 1.0709,   -inf,   -inf,   -inf],
        [2.2735, 2.1446, 2.2373, 2.2774, 2.5950, 0.3950,   -inf,   -inf],
        [2.9127, 2.3247, 1.4561, 1.2359, 1.5532, 2.1701, 4.3129,   -inf],
        [0.9512, 1.5161, 0.9081, 0.6172, 1.0501, 1.3023, 0.5608, 1.7355]])

Attention probabilities, head 0:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3418, 0.6582, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2199, 0.4767, 0.3034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1099, 0.3659, 0.2700, 0.2542, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1179, 0.2127, 0.2052, 0.2445, 0.2197, 0.000

### Step 11: Weighted sum of V
Multiply the attention probabilities by the (repeated) V: `[1, 9, T, T] @ [1, 9, T, 64] → [1, 9, T, 64]`. For each head, each token's output is a mix of the value vectors of the tokens it attends to, weighted by attention.

In [22]:
# Weighted sum of V (printing head 0, last token)

with torch.no_grad():
    attn_output = torch.matmul(
        attention_probs,
        repeated_v
    )

print("Attention probabilities shape:", attention_probs.shape)
print("Repeated V shape:", repeated_v.shape)
print("Attention output shape:", attn_output.shape)

print(f"\nHead 0 output for last token {last_tok!r}:")
print(attn_output[0, 0, last])

print("\nFirst 10 dimensions:")
print(attn_output[0, 0, last, :10])

Attention probabilities shape: torch.Size([1, 9, 8, 8])
Repeated V shape: torch.Size([1, 9, 8, 64])
Attention output shape: torch.Size([1, 9, 8, 64])

Head 0 output for last token ' lazy':
tensor([-0.0129,  0.0136, -0.0004, -0.0065,  0.0096, -0.0072, -0.0879, -0.0055,
         0.0101,  0.0016,  0.0022,  0.0022, -0.0094,  0.0031,  0.0004,  0.0001,
        -0.0185,  0.0034, -0.0015,  0.0197, -0.0139,  0.0050, -0.0039,  0.0013,
        -0.0020, -0.0089, -0.0132, -0.0031,  0.0004, -0.0004,  0.0039, -0.0045,
        -0.0112,  0.0021,  0.0031, -0.0047, -0.0080,  0.0110, -0.0061, -0.0008,
        -0.0110,  0.0181, -0.0014, -0.0026, -0.0380, -0.0013,  0.0048,  0.1102,
         0.0017,  0.0204, -0.0032,  0.0059,  0.0148, -0.0059,  0.0117,  0.0036,
         0.0152,  0.0029,  0.0076,  0.0033,  0.0067, -0.0006,  0.0101,  0.0054])

First 10 dimensions:
tensor([-0.0129,  0.0136, -0.0004, -0.0065,  0.0096, -0.0072, -0.0879, -0.0055,
         0.0101,  0.0016])


### Step 12: Merge heads + output projection
Undo the head split: transpose `[B, 9, T, 64] → [B, T, 9, 64]`, then reshape to `[B, T, 576]`. The learned `o_proj` then mixes information across heads. The result is the attention sub-layer's output.

*Note:* both the "After transpose" and "After reshape" lines print `combined_heads` after the reshape, so they show the same shape. The actual post-transpose shape is `[1, T, 9, 64]`.

In [23]:
# Combine the 9 attention heads back into the 576-dimensional representation

with torch.no_grad():
    # [B, 9, T, 64] -> [B, T, 9, 64]
    combined_heads = attn_output.transpose(1, 2).contiguous()

    # [B, T, 9, 64] -> [B, T, 576]
    combined_heads = combined_heads.reshape(
        *input_shape,
        -1
    )

    # Actual learned output projection
    projected_output = attn.o_proj(combined_heads)

print("After transpose:", combined_heads.shape)
print("After reshape:", combined_heads.shape)
print("After o_proj:", projected_output.shape)

print(f"\nAttention output for last token {last_tok!r}, first 10 dimensions:")
print(projected_output[0, last, :10])

After transpose: torch.Size([1, 8, 576])
After reshape: torch.Size([1, 8, 576])
After o_proj: torch.Size([1, 8, 576])

Attention output for last token ' lazy', first 10 dimensions:
tensor([-0.0105, -0.0070, -0.0043, -0.0106,  0.0641,  0.0244,  0.0613, -0.0919,
         0.0280,  0.0580])


### Check: compare with the model's attention module
Run the real `LlamaAttention` module on the same inputs and compare. The values for the last token match ours exactly.

**Caveat:** an all-zero mask is passed here, so this call applies **no causal masking**. It matches only because we compare the *last* token, and the last token is allowed to see every position anyway. Rows for earlier tokens would differ. The next cell checks the causal path properly. (Section 6 uses the same fact: a single new token in a decode step needs no mask.)

In [24]:
# Compare our manual attention calculation with the model's actual attention module

with torch.no_grad():
    actual_attn_output, _ = attn(
        hidden_states=normed_states,
        position_embeddings=(cos, sin),
        attention_mask=masked_scores.new_zeros(
            (1, 1, T, T)
        ),  # temporary; we'll verify the causal path separately
        past_key_values=None,
    )

print("Our reconstructed output shape:", projected_output.shape)
print("Actual attention output shape:", actual_attn_output.shape)

print(f"\nOur output, last token {last_tok!r}, first 10:")
print(projected_output[0, last, :10])

print(f"\nActual output, last token {last_tok!r}, first 10:")
print(actual_attn_output[0, last, :10])

Our reconstructed output shape: torch.Size([1, 8, 576])
Actual attention output shape: torch.Size([1, 8, 576])

Our output, last token ' lazy', first 10:
tensor([-0.0105, -0.0070, -0.0043, -0.0106,  0.0641,  0.0244,  0.0613, -0.0919,
         0.0280,  0.0580])

Actual output, last token ' lazy', first 10:
tensor([-0.0105, -0.0070, -0.0043, -0.0106,  0.0641,  0.0244,  0.0613, -0.0919,
         0.0280,  0.0580])


### Check: compare with `eager_attention_forward` (with the causal mask)
A stricter check. Convert our 2D boolean mask into the additive 4D mask HF expects (`0` = keep, `-inf` = block), then call the exact HF kernel. We pass the **un-repeated 3-head** K/V because `repeat_kv` runs inside the kernel.

Our output (before `o_proj`) matches for **every token and head**, with a maximum difference of `0.0`. Steps 7–11 reproduce HF attention exactly.

In [25]:
# Compare our existing manual attention output
# against the exact Hugging Face eager_attention_forward.
# No Block 0 recalculation.

from transformers.models.llama.modeling_llama import eager_attention_forward

with torch.no_grad():

    # Convert our existing 2D causal mask to the 4D
    # format expected by eager_attention_forward.
    causal_mask_4d = torch.zeros(
        (1, 1, T, T),
        device=masked_scores.device,
        dtype=masked_scores.dtype
    ).masked_fill(
        causal_mask.unsqueeze(0).unsqueeze(0),
        float("-inf")
    )

    # Exact HF implementation.
    # Pass the original 3 KV heads because repeat_kv()
    # happens inside eager_attention_forward().
    hf_attention_output, _ = eager_attention_forward(
        attn,
        rotated_q,
        rotated_k,
        value_states,
        causal_mask_4d,
        scaling=attn.scaling,
        dropout=0.0,
    )

    # Our existing manual result is [B, 9, T, 64].
    # HF returns [B, T, 9, 64].
    our_attention_output = attn_output.transpose(1, 2).contiguous()

    diff = (our_attention_output - hf_attention_output).abs()

print("Our output shape:", our_attention_output.shape)
print("HF output shape: ", hf_attention_output.shape)

print("\nMaximum difference:", diff.max().item())
print("Mean difference:   ", diff.mean().item())

print(
    "Equal within 1e-6:",
    torch.allclose(
        our_attention_output,
        hf_attention_output,
        atol=1e-6
    )
)

Our output shape: torch.Size([1, 8, 9, 64])
HF output shape:  torch.Size([1, 8, 9, 64])

Maximum difference: 0.0
Mean difference:    0.0
Equal within 1e-6: True


---
## 4. Finish block 0: residual + MLP

### Step 13: First residual + post-attention RMSNorm
Add the attention output back to the block input (`hidden_states`, the raw embeddings). This is residual `R1 = x + Attn(RMSNorm(x))`. Then apply `post_attention_layernorm` to get the MLP input.

In [26]:
# First residual after attention

with torch.no_grad():
    residual1_b0 = hidden_states + projected_output

    mlp_input_b0 = block0.post_attention_layernorm(residual1_b0)

print("Attention output:", projected_output.shape)
print("Residual R1:", residual1_b0.shape)
print("MLP input after RMSNorm:", mlp_input_b0.shape)

Attention output: torch.Size([1, 8, 576])
Residual R1: torch.Size([1, 8, 576])
MLP input after RMSNorm: torch.Size([1, 8, 576])


### Step 14: MLP gate branch
`gate_proj` expands 576 → 1536, then the SiLU activation (`x · sigmoid(x)`) is applied. This branch decides how much of each intermediate feature passes through.

In [27]:
# MLP — Gate branch + SiLU activation

with torch.no_grad():
    gate_b0 = block0.mlp.gate_proj(mlp_input_b0)
    gate_activated_b0 = block0.mlp.act_fn(gate_b0)

print("gate_proj output:", gate_b0.shape)
print("After SiLU:", gate_activated_b0.shape)

gate_proj output: torch.Size([1, 8, 1536])
After SiLU: torch.Size([1, 8, 1536])


### Step 15: MLP up branch
`up_proj` also expands 576 → 1536, with no activation. This branch carries the values to be gated.

In [28]:
# MLP — Up branch

with torch.no_grad():
    up_b0 = block0.mlp.up_proj(mlp_input_b0)

print("up_proj output:", up_b0.shape)

up_proj output: torch.Size([1, 8, 1536])


### Step 16: Gating (SwiGLU)
Multiply the two branches element-wise: `SiLU(gate) × up`. This is the "GLU" in SwiGLU. The activated gate scales each up-projected feature.

In [29]:
# MLP — Gated activation

with torch.no_grad():
    mlp_intermediate_b0 = gate_activated_b0 * up_b0

print("Gate × Up output:", mlp_intermediate_b0.shape)

Gate × Up output: torch.Size([1, 8, 1536])


### Step 17: Down projection
`down_proj` brings the 1536-dim intermediate back to the 576-dim model width. This is the MLP sub-layer's output.

In [30]:
# MLP — Down projection

with torch.no_grad():
    mlp_output_b0 = block0.mlp.down_proj(mlp_intermediate_b0)

print("down_proj output:", mlp_output_b0.shape)

down_proj output: torch.Size([1, 8, 576])


### Step 18: Second residual → Block 0 output
Add the MLP output to `R1`: `block0_output = R1 + MLP(RMSNorm(R1))`. This completes decoder layer 0. The shape `[1, T, 576]` matches the input, so it can go straight into layer 1.

In [31]:
# Block 0 — final residual connection

with torch.no_grad():
    block0_output = residual1_b0 + mlp_output_b0

print("R1:", residual1_b0.shape)
print("MLP output:", mlp_output_b0.shape)
print("Block 0 output:", block0_output.shape)

R1: torch.Size([1, 8, 576])
MLP output: torch.Size([1, 8, 576])
Block 0 output: torch.Size([1, 8, 576])


### Summary: shapes through block 0
All the intermediate tensors in one place. Every tensor stays at `[1, T, 576]` except the MLP intermediate, which briefly widens to 1536:

`embeddings → +attention (R1) → RMSNorm → MLP (576→1536→576) → +MLP → block 0 output`

In [32]:
print("hidden_states:", hidden_states.shape)
print("projected_output:", projected_output.shape)
print("residual1_b0:", residual1_b0.shape)
print("mlp_input_b0:", mlp_input_b0.shape)
print("mlp_intermediate_b0:", mlp_intermediate_b0.shape)
print("mlp_output_b0:", mlp_output_b0.shape)
print("block0_output:", block0_output.shape)

hidden_states: torch.Size([1, 8, 576])
projected_output: torch.Size([1, 8, 576])
residual1_b0: torch.Size([1, 8, 576])
mlp_input_b0: torch.Size([1, 8, 576])
mlp_intermediate_b0: torch.Size([1, 8, 1536])
mlp_output_b0: torch.Size([1, 8, 576])
block0_output: torch.Size([1, 8, 576])


---
## 5. Layers 1–28

### Run layers 1–28 with the real `layer.forward()`
Now that layer 0 is verified by hand, feed its output through layers 1–28 using the model's own decoder-layer forward. Every layer reuses the same RoPE `cos`/`sin` and the same 4D causal mask. `past_key_values=None` means no KV cache is built. This is an ordinary full-prompt pass over all T tokens.

`LAST_LAYER = num_hidden_layers - 1` (29 here), and `range(1, LAST_LAYER)` stops **before** it on purpose. The last layer is done by hand in Section 6.

In [33]:
LAST_LAYER = model.config.num_hidden_layers - 1   # 29 for SmolLM2-135M; done by hand in Section 6

hidden_states_1_28 = block0_output

with torch.no_grad():
    for layer_idx in range(1, LAST_LAYER):
        layer = model.model.layers[layer_idx]

        hidden_states_1_28 = layer(
            hidden_states=hidden_states_1_28,
            position_embeddings=(cos, sin),
            attention_mask=causal_mask_4d,
            past_key_values=None,
        )

print(f"Output after Block {LAST_LAYER - 1}:", hidden_states_1_28.shape)

Output after Block 28: torch.Size([1, 8, 576])


---
## 6. Layer 29 as a KV-cache decode step

Instead of processing all T tokens again, run the last layer the way it works during generation. Treat tokens `0 … last-1` as already processed, with K/V **in the cache**, and treat the last token (`last_tok`: `six` in the counting prompt, `lazy` in the fox prompt) as the **new token**. Only the new token needs Q, and only its row of attention is computed.

This works because of the causal mask. The layer-29 input for tokens `0 … last-1` does not depend on the last token, so the K/V computed here for them are exactly what a real cache would hold.

### Step 19: Build the "cache", K/V for tokens `0 … last-1`
Apply layer 29's `input_layernorm` to the first `last` (= T-1) hidden states, then run **only** `k_proj` and `v_proj`. Cached tokens never need Q again. Reshape into 3 KV heads and apply RoPE to K at positions `0 … last-1`. `apply_rotary_pos_emb` rotates Q and K together, so a `zeros_like` placeholder stands in for Q and is thrown away.

The result, `[1, 3, T-1, 64]` for K and for V (`k_cached` / `v_cached`), is what `past_key_values` stores for this layer: 3 KV heads, not 9.

In [34]:
# Block 29 — cached tokens (all but the last)
# Calculate K/V for tokens 0..last-1

block29 = model.model.layers[LAST_LAYER]
attn29 = block29.self_attn

cached_hidden = hidden_states_1_28[:, :last, :]

with torch.no_grad():
    normed_cached = block29.input_layernorm(cached_hidden)

    k_cached = attn29.k_proj(normed_cached)
    v_cached = attn29.v_proj(normed_cached)

    k_cached = k_cached.view(
        1, last, -1, attn29.head_dim
    ).transpose(1, 2)

    v_cached = v_cached.view(
        1, last, -1, attn29.head_dim
    ).transpose(1, 2)

    # Positions 0..last-1
    cos_cached = cos[:, :last, :]
    sin_cached = sin[:, :last, :]

    # RoPE is applied to K before it is reused.
    _, k_cached = apply_rotary_pos_emb(
        torch.zeros_like(k_cached),
        k_cached,
        cos_cached,
        sin_cached,
    )

print("K for cached tokens:", k_cached.shape)
print("V for cached tokens:", v_cached.shape)

K for cached tokens: torch.Size([1, 3, 7, 64])
V for cached tokens: torch.Size([1, 3, 7, 64])


### Step 20: Q/K/V for the new token only
Take just the last position's hidden state (`[:, last:T, :]`, shape `[1, 1, 576]`), normalize it, and project to Q, K and V. The sequence dimension is now **1**:
- Q → `[1, 9, 1, 64]`
- K, V → `[1, 3, 1, 64]`

This is where decoding saves compute. The projections run for one token, not the whole prompt.

In [35]:
# Block 29 — new token only (the last prompt token)

new_token_hidden = hidden_states_1_28[:, last:T, :]

with torch.no_grad():
    normed_new_token = block29.input_layernorm(new_token_hidden)

    q_new = attn29.q_proj(normed_new_token)
    k_new = attn29.k_proj(normed_new_token)
    v_new = attn29.v_proj(normed_new_token)

    q_new = q_new.view(
        1, 1, -1, attn29.head_dim
    ).transpose(1, 2)

    k_new = k_new.view(
        1, 1, -1, attn29.head_dim
    ).transpose(1, 2)

    v_new = v_new.view(
        1, 1, -1, attn29.head_dim
    ).transpose(1, 2)

print("Q new token:", q_new.shape)
print("K new token:", k_new.shape)
print("V new token:", v_new.shape)

Q new token: torch.Size([1, 9, 1, 64])
K new token: torch.Size([1, 3, 1, 64])
V new token: torch.Size([1, 3, 1, 64])


### Step 21: RoPE at the new token's real position
The new token must be rotated at its **absolute** position, `last` (= T-1: 5 for the counting prompt, 7 for the fox prompt), not position 0, even though it is the only token in this batch. In real generation, `position_ids` for the new token equals the current cache length. Getting this wrong is a classic KV-cache bug: the new token's Q and K would carry the wrong position.

In [36]:
# Block 29 — RoPE for the new token

position_ids_new = torch.tensor(
    [[last]],
    device=new_token_hidden.device
)

with torch.no_grad():
    cos_new, sin_new = model.model.rotary_emb(
        new_token_hidden,
        position_ids=position_ids_new,
    )

    q_new, k_new = apply_rotary_pos_emb(
        q_new,
        k_new,
        cos_new,
        sin_new,
    )

print("RoPE Q new token:", q_new.shape)
print("RoPE K new token:", k_new.shape)
print("V new token:", v_new.shape)

RoPE Q new token: torch.Size([1, 9, 1, 64])
RoPE K new token: torch.Size([1, 3, 1, 64])
V new token: torch.Size([1, 3, 1, 64])


### Step 22: Append to the cache + GQA expansion
Concatenate the cached K/V (positions `0 … last-1`) with the new token's K/V (position `last`) along the sequence dimension to get `[1, 3, T, 64]`. This is what `past_key_values.update(...)` does inside `LlamaAttention.forward`. The 3-head `k_all` / `v_all` is the updated cache.

Then use `repeat_kv` to expand K/V to 9 heads for the matmul, as in Step 8. The expansion is not stored.

In [37]:
# Block 29 — complete K/V sequence for the new-token attention

with torch.no_grad():
    # Previous K/V + newly calculated K/V
    k_all = torch.cat([k_cached, k_new], dim=2)
    v_all = torch.cat([v_cached, v_new], dim=2)

    # GQA: 3 KV heads → 9 attention heads
    k_all_repeated = repeat_kv(
        k_all,
        attn29.num_key_value_groups
    )

    v_all_repeated = repeat_kv(
        v_all,
        attn29.num_key_value_groups
    )

print("Q (new token):", q_new.shape)
print("K (all positions):", k_all.shape)
print("V (all positions):", v_all.shape)
print("Repeated K:", k_all_repeated.shape)
print("Repeated V:", v_all_repeated.shape)

Q (new token): torch.Size([1, 9, 1, 64])
K (all positions): torch.Size([1, 3, 8, 64])
V (all positions): torch.Size([1, 3, 8, 64])
Repeated K: torch.Size([1, 9, 8, 64])
Repeated V: torch.Size([1, 9, 8, 64])


### Step 23: Scores for one query against all keys
`Q_last · Kᵀ · scale` gives `[1, 9, 1, T]`: **one row** per head instead of the T×T matrix from Step 9. During decoding, each new token costs one row of attention, so the work grows linearly with context length instead of recomputing the full square.

In [38]:
# Block 29 — attention scores for the new token

with torch.no_grad():
    attention_scores_new = torch.matmul(
        q_new,
        k_all_repeated.transpose(2, 3)
    ) * attn29.scaling

print("Q (new token):", q_new.shape)
print("Kᵀ:", k_all_repeated.transpose(2, 3).shape)
print("Attention scores:", attention_scores_new.shape)

Q (new token): torch.Size([1, 9, 1, 64])
Kᵀ: torch.Size([1, 9, 64, 8])
Attention scores: torch.Size([1, 9, 1, 8])


### Step 24: Mask, softmax, weighted V, `o_proj`
- **Mask:** the "causal mask" here is all zeros. The newest token may see every position `0 … last`, so nothing needs blocking. In a decode step, the cache contains only the past.
- **Softmax** runs in `float32` and is cast back, exactly as `eager_attention_forward` does.
- **Weighted sum of V**, then merge the heads `[1, 9, 1, 64] → [1, 1, 576]` and apply `o_proj`.

The result is layer 29's attention output for the new token only.

In [39]:
# Block 29 — finish attention for the new token

with torch.no_grad():
    # Causal mask: the new token can attend to every position 0..last
    attention_scores_new = attention_scores_new + torch.zeros(
        (1, 1, 1, T),
        device=attention_scores_new.device,
        dtype=attention_scores_new.dtype,
    )

    # Softmax → attention probabilities
    attention_probs_new = torch.softmax(
        attention_scores_new,
        dim=-1,
        dtype=torch.float32,
    ).to(q_new.dtype)

    # Weighted sum of V
    attention_output_new = torch.matmul(
        attention_probs_new,
        v_all_repeated,
    )

    # [B, heads, 1, head_dim]
    #       ↓
    # [B, 1, heads, head_dim]
    attention_output_new = attention_output_new.transpose(1, 2).contiguous()

    # [B, 1, 9, 64] → [B, 1, 576]
    attention_output_new = attention_output_new.reshape(
        1, 1, -1
    )

    # Final attention projection
    projected_output_new = attn29.o_proj(attention_output_new)

print("Attention probabilities:", attention_probs_new.shape)
print("Attention output:", attention_output_new.shape)
print("After o_proj:", projected_output_new.shape)

Attention probabilities: torch.Size([1, 9, 1, 8])
Attention output: torch.Size([1, 1, 576])
After o_proj: torch.Size([1, 1, 576])


### Step 25: Finish layer 29 → final norm → LM head → next token
Complete the block for the new token. The first residual adds `new_token_hidden` (the pre-norm input) and the attention output. Then `post_attention_layernorm` → MLP → second residual. After the last layer, apply the model's final `norm` and project to the 49,152-token vocabulary with `lm_head`.

The argmax is the same prediction as the full `model(**inputs)` baseline at the start of Section 3: **`' seven'` (id 4962)** for the counting prompt and **`' dog'` (id 2767)** for the fox prompt. The hand-built pipeline, including the KV-cache decode step, reproduces the model end to end.

Matching the top token is a weak test, so the next cell compares the full logit vector.

In [40]:
# Final step — Block 29 → final RMSNorm → LM head → next token

with torch.no_grad():
    # Complete Block 29 for the new token
    residual29 = new_token_hidden + projected_output_new

    # For this final demonstration, run the remaining MLP of Block 29
    mlp_input29 = block29.post_attention_layernorm(residual29)
    mlp_output29 = block29.mlp(mlp_input29)

    block29_output_new = residual29 + mlp_output29

    # Final model RMSNorm
    final_hidden = model.model.norm(block29_output_new)

    # Vocabulary logits
    logits = model.lm_head(final_hidden)

    # Highest-probability next token
    next_token_id = logits[:, -1, :].argmax(dim=-1)
    next_token = tokenizer.decode(next_token_id)

print("Next token ID:", next_token_id.item())
print("Next token:", repr(next_token))

Next token ID: 2767
Next token: ' dog'


### Final check: our logits vs. the model's logits
Run the full `model(**inputs)` forward again and compare the **entire 49,152-dim logit vector** for the last position, not just the argmax.

Results for the two example prompts:

| Prompt | `allclose(atol=1e-4)` | Max abs difference | Our token | Model token |
|---|---|---|---|---|
| Counting (T = 6) | **True** | ≈ 3.2e-5 | `' seven'` | `' seven'` |
| Fox (T = 8) | **True** | ≈ 3.4e-5 | `' dog'` | `' dog'` |

The differences are float32 rounding noise. The operations run in a different order (a 1-token decode step vs. one full-prompt pass, separate matmuls vs. fused calls), so tiny differences are expected. The check also passed with 3- and 9-token prompts (max difference around 2e-5).

This confirms the whole hand-built path is numerically equivalent to Hugging Face's forward pass. That includes layer 0 by hand, layers 1–28 through `layer.forward()`, and layer 29 as a KV-cache decode step. Reusing cached K/V for earlier tokens and computing only the new token's Q/K/V gives the same answer as recomputing everything.

In [41]:
# Final verification — compare our logits with the model's logits directly

with torch.no_grad():
    outputs = model(**inputs)

print(
    "Logits allclose:",
    torch.allclose(
        logits[0, -1],
        outputs.logits[0, -1],
        atol=1e-4
    )
)

print(
    "Maximum logit difference:",
    (logits[0, -1] - outputs.logits[0, -1]).abs().max().item()
)

print("Our predicted token:  ", repr(tokenizer.decode(logits[0, -1].argmax())))
print("Model predicted token:", repr(tokenizer.decode(outputs.logits[0, -1].argmax())))

Logits allclose: True
Maximum logit difference: 3.4332275390625e-05
Our predicted token:   ' dog'
Model predicted token: ' dog'
